In [45]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
load_dotenv()
if os.environ['OPENAI_API_KEY']:
    print("API key available")
llm=ChatGroq(model="llama-3.3-70b-versatile",temperature=0.7)  

API key available


## RAG WITH TEXT

#### 1: Preparing Document for text

In [13]:
from langchain_core.documents import Document
text_data= """Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers have used techniques including state space search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind and Meta, aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human."""

docs=[Document(page_content=text_data,metadata={"source":"Wikipedia","Document_id":"Doc1"})]
docs

[Document(metadata={'source': 'Wikipedia', 'Document_id': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]\n\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.\n\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language 

#### 2: Chunking

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter= RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks=splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'Wikipedia', 'Document_id': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'Wikipedia', 'Document_id': 'Doc1'}, page_content='High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.'),
 Document(metadata={'source': 'Wikiped

#### 3: Creating Embeddings

In [46]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4975.57it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [38]:
embedding_model.embed_query("hello")


[0.03063984215259552,
 -0.0062302108854055405,
 -0.002121538622304797,
 0.013879180885851383,
 0.026486821472644806,
 0.00406580651178956,
 -0.0038712257519364357,
 0.03526987135410309,
 0.011488322168588638,
 -0.0038065204862505198,
 0.013236375525593758,
 -0.0118856867775321,
 -0.00016034094733186066,
 0.02157069928944111,
 0.03014962002635002,
 -0.10559132695198059,
 0.03541700541973114,
 0.006596212740987539,
 -0.05530388280749321,
 -0.00022679209359921515,
 -0.007831409573554993,
 -0.004739568568766117,
 -2.195034539909102e-05,
 0.03705041483044624,
 0.02242698334157467,
 0.004295800346881151,
 -0.0010749677894636989,
 0.00741618825122714,
 0.011647233739495277,
 0.019009584560990334,
 -0.011946453712880611,
 0.002173912012949586,
 0.026941770687699318,
 0.01608094573020935,
 2.1605060283036437e-06,
 -0.008001395501196384,
 -0.013405372388660908,
 0.016731999814510345,
 -0.009559371508657932,
 -0.022413142025470734,
 -0.000163130447617732,
 -0.0004943632520735264,
 -0.009095543064

#### 4: Creating and storing embeddings in vector db

In [39]:
from langchain_community.vectorstores import Chroma
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [ ]:
# #BTS of above code (langchain did this)

# vectors=[]
# for docs in chunks:
#     vector=embedding_model.embed_documents(docs.page_content)
#     vectors.append(vector) 
#chromadb store vector + metadata 

#### Sementic search

In [41]:
vectorstore.similarity_search("What is ai",k=3)

[Document(metadata={'Document_id': 'Doc1', 'source': 'Wikipedia'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]'),
 Document(metadata={'Document_id': 'Doc1', 'source': 'Wikipedia'}, page_content='The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers have used techniques including state space search and mathematical optimization, formal logic, artificial neural networ

In [47]:
context = vectorstore.similarity_search("what is ai?")

In [48]:
response=llm.invoke(f"What is ai? answer using following context:{context}")
print(response.content)

Based on the provided context from Wikipedia, Artificial Intelligence (AI) refers to the capability of computational systems to perform tasks that are typically associated with human intelligence. These tasks include:

1. **Learning**: The ability to acquire new knowledge or skills.
2. **Reasoning**: The ability to draw inferences, make decisions, and solve problems.
3. **Problem-solving**: The ability to find solutions to complex problems.
4. **Perception**: The ability to interpret and understand data from the environment.
5. **Decision-making**: The ability to make choices based on available data and goals.

AI is a field of research that combines engineering, mathematics, and computer science to develop methods and software that enable machines to:

1. **Perceive their environment**: Understand and interpret data from the environment.
2. **Use learning and intelligence**: Apply knowledge and skills to take actions.
3. **Maximize chances of achieving defined goals**: Make decisions 